# Video Data Pipeline with Lance

Data touches everything in video generation -- re-captioning billions of samples, crafting data for real-time interaction, causality, multi-shot. This notebook builds a video data pipeline using Lance, a data format used in production by leading video generation labs. Lance: 100x faster random access than Parquet, native video ops (seek, decode windows, skip frames), built-in vector indexing. ~3-4 hrs.

## Why Lance?

Open lakehouse format for multimodal AI — not a vector database, but a **columnar data format with native vector support**. The distinction matters: instead of maintaining a separate vector search service (FAISS, Milvus, Pinecone) alongside your data, vector operations live inside the same format that stores your features, metadata, and media.

Key advantages over Parquet:

1. **Random access 100x faster than Parquet** — critical for training with global shuffling.
2. **Native video:** seek to any frame, decode temporal windows, skip irrelevant frames.
3. **Built-in ANN vector index** (IVF_PQ) — no external vector DB needed. For <1M embeddings, even brute-force NumPy `@` matmul is fast enough (~50ms). Lance scales beyond that.
4. **Columnar + row-group format** scales to billions of rows with append-only writes.
5. **Zero-copy PyTorch integration** — embeddings go straight to tensors without serialization.

In [2]:
import cv2
import numpy as np
import torch
from pathlib import Path
from scenedetect import detect, ContentDetector
import lancedb
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

WORK_DIR = Path("./data/video_pipeline")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Download sample videos (public domain)
import requests

sample_urls = {
    "bunny.mp4": "https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/360/Big_Buck_Bunny_360_10s_1MB.mp4",
}
for name, url in sample_urls.items():
    path = WORK_DIR / name
    if not path.exists():
        print(f"Downloading {name}...")
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        path.write_bytes(r.content)
        print(f"  Saved to {path}")

# Verify the download
cap = cv2.VideoCapture(str(WORK_DIR / "bunny.mp4"))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"\nVideo info: {width}x{height}, {fps:.1f} fps, {total_frames} frames, {total_frames/fps:.1f}s")

Device: cpu
  Saved to data/video_pipeline/bunny.mp4

Video info: 640x360, 30.0 fps, 300 frames, 10.0s


In [ ]:
# Scene detection with PySceneDetect
# ContentDetector compares frame-to-frame changes in HSV color space.
# threshold=27.0 is the default; lower = more sensitive (more cuts detected).
# For production: also consider AdaptiveDetector for variable lighting.

from scenedetect import detect, ContentDetector, open_video

video_path = str(WORK_DIR / "bunny.mp4")
scene_list = detect(video_path, ContentDetector(threshold=27.0))

# If no scene changes detected (short clip), treat entire video as one scene
if len(scene_list) == 0:
    from scenedetect.frame_timecode import FrameTimecode

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_val = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    scene_list = [(FrameTimecode(0, fps=fps_val), FrameTimecode(total, fps=fps_val))]
    print("No scene cuts detected -- treating entire video as one scene.")

print(f"Detected {len(scene_list)} scenes:")
for i, (start, end) in enumerate(scene_list):
    n_frames = end.get_frames() - start.get_frames()
    print(f"  Scene {i}: {start.get_timecode()} - {end.get_timecode()} ({n_frames} frames)")

# Extract keyframes from each scene (middle frame)
keyframes = []
cap = cv2.VideoCapture(video_path)
for start, end in scene_list:
    mid_frame = (start.get_frames() + end.get_frames()) // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
    ret, frame = cap.read()
    if ret:
        keyframes.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

# Display keyframes
if keyframes:
    n_show = min(len(keyframes), 6)
    fig, axes = plt.subplots(1, n_show, figsize=(18, 3))
    if n_show == 1:
        axes = [axes]
    for ax, kf in zip(axes, keyframes[:6]):
        ax.imshow(kf)
        ax.axis("off")
    plt.suptitle("Scene Keyframes")
    plt.show()

print(f"\nExtracted {len(keyframes)} keyframes")

In [ ]:
# EXERCISE: Quality filtering — resolution check + motion estimation
# Two key quality signals:
#   1. Resolution: reject clips below minimum resolution
#   2. Motion: reject static clips (boring) and chaotic ones (corrupted/transitions)
# Use optical flow for dense motion estimation between sampled frame pairs.


def analyze_clip(video_path: str, start_frame: int, end_frame: int, sample_rate: int = 5) -> dict:
    """Analyze a video clip for quality metrics.

    Args:
        video_path: path to the video file
        start_frame: first frame index of the clip
        end_frame: last frame index of the clip
        sample_rate: analyze every Nth frame (skip frames for speed)

    Returns:
        dict with keys: width, height, num_frames, avg_motion,
        resolution_ok (bool), motion_ok (bool)
    """
    # YOUR CODE:
    # - Read video resolution from the capture
    # - Estimate motion via optical flow between consecutive sampled frames
    # - Return quality metrics including pass/fail booleans
    raise NotImplementedError


# Analyze each scene
clip_analyses = []
for i, (start, end) in enumerate(scene_list):
    analysis = analyze_clip(video_path, start.get_frames(), end.get_frames())
    analysis["scene_idx"] = i
    clip_analyses.append(analysis)
    status = "PASS" if (analysis["resolution_ok"] and analysis["motion_ok"]) else "FAIL"
    print(
        f"Scene {i}: {analysis['width']}x{analysis['height']}, "
        f"motion={analysis['avg_motion']:.2f}, "
        f"{status}"
    )

passed = sum(1 for c in clip_analyses if c["resolution_ok"] and c["motion_ok"])
print(f"\n{passed}/{len(clip_analyses)} clips passed quality filter")

In [ ]:
# Motion analysis histogram
# This visualization helps tune the motion thresholds.
# Too strict = not enough training data. Too loose = garbage in.

motions = [c["avg_motion"] for c in clip_analyses]

plt.figure(figsize=(10, 4))
plt.hist(motions, bins=max(20, len(motions)), edgecolor="black", alpha=0.7)
plt.axvline(0.5, color="red", linestyle="--", label="Min threshold (0.5)")
plt.axvline(50, color="red", linestyle="--", label="Max threshold (50)")
plt.xlabel("Average Optical Flow Magnitude")
plt.ylabel("Count")
plt.title("Motion Distribution Across Clips")
plt.legend()
plt.show()

print(f"Motion stats: min={min(motions):.2f}, max={max(motions):.2f}, "
      f"mean={np.mean(motions):.2f}, median={np.median(motions):.2f}")

In [ ]:
# Caption generation using BLIP
# In production (at scale): use a better model (BLIP-2, LLaVA, CogVLM)
# and run as a batch GPU job across millions of clips.
# At scale: re-captioning on the order of billions of samples
# GPU memory: ~1 GB for BLIP-base in FP16.

from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base", torch_dtype=torch.float16
).to(device)

captions = []
for i, kf in enumerate(tqdm(keyframes, desc="Captioning")):
    pil_img = Image.fromarray(kf)
    inputs = processor(pil_img, return_tensors="pt").to(device, torch.float16)
    out = blip_model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(out[0], skip_special_tokens=True)
    captions.append(caption)
    print(f"  Scene {i}: {caption}")

del blip_model, processor
torch.cuda.empty_cache()
print(f"\nGenerated {len(captions)} captions")

In [ ]:
# CLIP embedding computation
# These embeddings enable semantic search over the dataset.
# L2-normalized so cosine similarity = dot product.
# GPU memory: ~0.5 GB for CLIP ViT-B/16.

from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

embeddings = []
for kf in tqdm(keyframes, desc="Embedding"):
    pil_img = Image.fromarray(kf)
    inputs = clip_processor(images=pil_img, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)  # L2 normalize
    embeddings.append(emb.cpu().numpy().flatten())

print(f"Computed {len(embeddings)} embeddings, shape: {embeddings[0].shape}")

# Keep clip_model loaded for query encoding later
# del clip_model, clip_processor
# torch.cuda.empty_cache()

In [ ]:
# EXERCISE: Store in Lance format
# Lance is a columnar format optimized for ML workloads:
#   - Append-only writes (no rewrite on update)
#   - Fragment-based storage (parallel reads)
#   - Native vector column type
#   - Built-in ANN index (IVF_PQ)

import lancedb
import pyarrow as pa

db = lancedb.connect(str(WORK_DIR / "video_lance_db"))

# YOUR CODE: Build records list — each clip becomes a row
# Each record should include: scene_idx, video_path, start/end frames,
# resolution, quality metrics, caption, and embedding vector.
# Then create a Lance table from the records.
#
# Available data: clip_analyses (list of dicts), captions (list of str),
# embeddings (list of np arrays), scene_list (list of (start, end) timecodes)

records = []
raise NotImplementedError

table = db.create_table("clips", records, mode="overwrite")
print(f"Stored {len(records)} clips in Lance")
print(f"\nSchema:")
print(table.schema)
print(f"\nTable size: {table.count_rows()} rows")

In [ ]:
# Vector search over embeddings — inside the data format, no external service
# Encode a text query with CLIP, then find nearest clips by embedding distance.
# This is how you'd build a data curation UI or find similar training examples.
# Key insight: this vector search runs on the same Lance table that stores your
# metadata, captions, and quality scores — no separate FAISS/Milvus/Pinecone needed.

query_text = "animal in nature"
inputs = clip_processor(text=[query_text], return_tensors="pt").to(device)
with torch.no_grad():
    query_emb = clip_model.get_text_features(**inputs)
    query_emb = query_emb / query_emb.norm(dim=-1, keepdim=True)

results = table.search(query_emb.cpu().numpy().flatten()).limit(3).to_pandas()
print(f"Top 3 results for '{query_text}':")
print(results[["scene_idx", "caption", "_distance"]].to_string())

# Visualize the top results
if len(results) > 0:
    n_results = min(len(results), 3)
    fig, axes = plt.subplots(1, n_results, figsize=(6 * n_results, 4))
    if n_results == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, results.iterrows()):
        idx = int(row["scene_idx"])
        if idx < len(keyframes):
            ax.imshow(keyframes[idx])
            ax.set_title(f"Scene {idx}: {row['caption']}\ndist={row['_distance']:.3f}", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"Query: '{query_text}'")
    plt.tight_layout()
    plt.show()

In [ ]:
# Lance-native operations demo
# These operations are what make Lance 100x faster than Parquet for ML training.

# Fast random access -- read specific rows by index (O(1), not O(n))
df = table.to_pandas()
row = df.iloc[0]
print(f"Random access row 0: scene {row['scene_idx']}, caption: '{row['caption']}'")

# Filter + scan (predicate pushdown -- only reads relevant fragments)
filtered = table.search().where("avg_motion > 1.0").to_pandas()
print(f"\nHigh-motion clips (avg_motion > 1.0): {len(filtered)}")

# Combine vector search with metadata filters
filtered_search = (
    table.search(query_emb.cpu().numpy().flatten())
    .where("resolution_ok = true")
    .limit(3)
    .to_pandas()
)
print(f"\nVector search + quality filter: {len(filtered_search)} results")
if len(filtered_search) > 0:
    print(filtered_search[["scene_idx", "caption", "_distance"]].to_string())

print("\n--- Why this matters for training ---")
print("1. Random access enables global shuffling (Parquet requires sequential scan)")
print("2. Columnar reads: only load columns you need (skip embeddings during captioning jobs)")
print("3. Vector search without external index (no separate FAISS/Milvus service)")
print("4. Predicate pushdown: filter at storage level, not in Python")

In [ ]:
# EXERCISE: PyTorch DataLoader backed by Lance
# This is the pattern for training: Lance table -> Dataset -> DataLoader.
# Key advantage: fast random access enables proper global shuffling.

from torch.utils.data import Dataset, DataLoader


class LanceVideoDataset(Dataset):
    """PyTorch Dataset backed by Lance — fast random access + global shuffling.

    Args:
        lance_table: a LanceDB table object
        transform: optional transform to apply to frame tensors

    __getitem__ should return: (frame_tensor, caption_str, embedding_array)
        - frame_tensor: (3, 256, 256) float tensor in [0, 1]
        - caption_str: the caption string for this clip
        - embedding_array: numpy array of the CLIP embedding
    """

    def __init__(self, lance_table, transform=None):
        self.df = lance_table.to_pandas()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # YOUR CODE:
        # Load the middle frame of the clip, convert to RGB, resize to 256x256,
        # convert to a float tensor in [0, 1] with shape (C, H, W).
        # Return (tensor, caption, embedding).
        raise NotImplementedError


dataset = LanceVideoDataset(table)
loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)

# Test iteration
batch = next(iter(loader))
frames, caps, embs = batch
print(f"Batch shapes: frames={frames.shape}, embeddings={embs.shape}")
print(f"Captions: {list(caps)}")

# Visualize the batch
fig, axes = plt.subplots(1, min(4, frames.shape[0]), figsize=(16, 4))
if not isinstance(axes, np.ndarray):
    axes = [axes]
for i, ax in enumerate(axes):
    if i < frames.shape[0]:
        ax.imshow(frames[i].permute(1, 2, 0).numpy())
        ax.set_title(f"{caps[i][:40]}...", fontsize=8)
    ax.axis("off")
plt.suptitle("Training Batch from Lance-backed DataLoader")
plt.show()

## Scaling: 100 Clips to 1 Billion

At production scale, Lance tables can grow to TB+ with efficient append-only writes. Vector index (IVF_PQ) handles billions of embeddings. Horizontal scaling via Lance's fragment-based storage.

Pipeline becomes: **ingest -> scene detect -> filter -> caption (batch GPU job) -> embed -> pack to Lance -> index.**

Example at scale: "30M videos, train classifier for annotation, run pipeline, pack, add to ongoing training run."

## The Feature Lakehouse Pattern

The pipeline above (ingest → filter → caption → embed → store) is a useful mental model, but production systems at scale use a **feature lakehouse** — a centralized, queryable store that evolves over time.

### Why not a linear pipeline?

At petabyte scale, you **cannot copy data** to construct new training datasets per experiment. Instead:

1. **Store once, query many times.** Raw video, embeddings, captions, quality scores, and derived features all live in the same Lance table as columns.
2. **Schema evolution.** Need to swap CLIP ViT-B for SigLIP? Append a new embedding column — don't rewrite the entire dataset. Lance supports column appending without touching existing data.
3. **Training jobs query in-place.** A training run specifies which columns it needs (e.g., "give me video frames + SigLIP embeddings + captions where aesthetic_score > 0.7 and motion_score > 1.0"). No intermediate dataset materialization.
4. **Semantic exploration.** Researchers search the lakehouse via vector queries ("find clips with camera pans over water"), not just SQL. The same embedding index serves both curation and training.

### Lance makes this possible

```python
# Schema evolution: append a new embedding column without rewriting
# (conceptual — real implementation uses Lance's merge/update API)
import lancedb

db = lancedb.connect("s3://training-data/lakehouse")
table = db.open_table("video_clips")

# Original schema: video_path, caption, clip_embedding, quality_score, ...
# New: append SigLIP embeddings alongside existing CLIP embeddings
table.add_columns({"siglip_embedding": new_siglip_vectors})

# Training query: only read what you need (columnar pushdown)
training_data = (
    table.search(query_embedding)          # vector search
    .where("quality_score > 0.7")          # metadata filter
    .select(["video_path", "siglip_embedding", "caption"])  # only these columns
    .limit(1_000_000)
    .to_batches()                           # stream to DataLoader
)
```

### Why this matters for system design

When designing a data pipeline, don't describe a linear flow that terminates in WebDataset shards. Describe a **lakehouse that training queries directly**:

- Shard-based formats (WebDataset, MDS) are for **serving pre-materialized datasets to distributed training**
- Lance lakehouse is for **the canonical data store** where you curate, explore, and version your data
- In practice, both coexist: Lance for the living dataset, WebDataset for frozen snapshots served to large training runs

## Data at Scale

"Re-captioning on the order of billions of samples, crafting data for real-time interaction, causality, multi-shot." This means:

1. **Existing captions are often wrong/low-quality** -- re-caption everything with better models.
2. **Different training objectives need different data curation** -- data for real-time interaction is different from data for quality generation.
3. **Multi-shot:** multiple clips from same scene/narrative need to be linked.

## Checkpoint

**How would you improve training data for a model with poor temporal consistency?**

1. **Curate:** filter for clips with smooth, consistent motion (optical flow analysis).
2. **Augment:** temporal jittering, speed variations to teach temporal robustness.
3. **Caption:** include temporal descriptions ("camera pans left", "person walks forward").
4. **Structure:** ensure training batches contain variable-length clips.
5. **Evaluate:** build temporal consistency metrics into the data pipeline feedback loop.

**Further reading:** [LanceDB docs](https://lancedb.github.io/lancedb/), [PySceneDetect docs](https://www.scenedetect.com/).